# FantasAI - Stage 6: ML Prediction Models - Phase 1: Data Preparation

## Overview
This notebook prepares data for training machine learning models to predict weekly fantasy points for NFL players.

## Objectives
1. **Load ML Features**: Import engineered features from `main.fantasai.ml_player_features`
2. **Add Context Features**: Opponent strength, game context, player usage patterns
3. **Create Temporal Splits**: Train/validation/test splits respecting temporal ordering
4. **Build Baselines**: Simple baseline models for comparison (season average, last-3-game-avg)
5. **Evaluate Baselines**: Establish performance benchmarks for ML models

## Data Split Strategy
```
Training:   2024 Weeks 1-12  (First 12 weeks of season)
Validation: 2024 Weeks 13-15 (Late season for tuning)
Test:       2024 Weeks 16-18 + 2025 data (Playoff weeks + new season)
```

## Success Criteria
* Train/val/test splits created with no data leakage
* Baseline RMSE established per position
* Feature distributions validated
* Missing data patterns identified

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
import numpy as np

# Configuration
catalog = "main"
schema = "fantasai"
source_table = f"{catalog}.{schema}.ml_player_features"

print("FantasAI ML Prediction - Phase 1: Data Preparation")
print("=" * 70)
print(f"Source table: {source_table}")
print()

# Display available features
print("Loading ML features...")
ml_features = spark.table(source_table)
print(f"✓ Loaded {ml_features.count():,} player-week records")
print(f"✓ Features: {len(ml_features.columns)} columns")
print()

# Show schema
print("Feature columns:")
for col in sorted(ml_features.columns):
    print(f"  - {col}")

In [0]:
print("Data Distribution Analysis")
print("=" * 70)
print()

# Overall statistics
print("Overall Statistics:")
overall_stats = ml_features.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("master_player_id").alias("unique_players"),
    F.countDistinct("position").alias("positions"),
    F.min("season").alias("min_season"),
    F.max("season").alias("max_season"),
    F.min("week").alias("min_week"),
    F.max("week").alias("max_week")
).first()

print(f"  Total Records: {overall_stats.total_records:,}")
print(f"  Unique Players: {overall_stats.unique_players:,}")
print(f"  Positions: {overall_stats.positions}")
print(f"  Season Range: {overall_stats.min_season} - {overall_stats.max_season}")
print(f"  Week Range: {overall_stats.min_week} - {overall_stats.max_week}")
print()

# Position distribution with target variable statistics
print("Position Distribution:")
position_stats = ml_features.groupBy("position").agg(
    F.count("*").alias("records"),
    F.countDistinct("master_player_id").alias("players"),
    F.mean("current_week_points").alias("avg_points"),
    F.stddev("current_week_points").alias("std_points"),
    F.min("current_week_points").alias("min_points"),
    F.max("current_week_points").alias("max_points")
).orderBy("position")

display(position_stats)

# Season and week distribution
print("\nSeason-Week Distribution:")
season_week_dist = ml_features.groupBy("season", "week").agg(
    F.count("*").alias("records"),
    F.countDistinct("master_player_id").alias("unique_players")
).orderBy("season", "week")

display(season_week_dist)

In [0]:
print("Feature Quality Assessment")
print("=" * 70)
print()

# Check for missing values in key features
print("Missing Value Analysis:")

# Get all numeric columns (excluding IDs and categorical)
numeric_cols = [f.name for f in ml_features.schema.fields 
                if f.dataType.typeName() in ['double', 'integer', 'long', 'float']
                and f.name not in ['season', 'week']]

# Calculate null counts
null_counts = ml_features.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in numeric_cols]
).first()

total_records = ml_features.count()
missing_summary = []

for col in numeric_cols:
    null_count = null_counts[col]
    if null_count > 0:
        missing_pct = (null_count / total_records) * 100
        missing_summary.append((col, null_count, missing_pct))

if missing_summary:
    print(f"  Found {len(missing_summary)} features with missing values:\n")
    for col, count, pct in sorted(missing_summary, key=lambda x: x[1], reverse=True)[:10]:
        print(f"    {col}: {count:,} ({pct:.2f}%)")
else:
    print("  ✓ No missing values found in numeric features")

print()

# Target variable distribution
print("Target Variable (current_week_points) Distribution:")
target_stats = ml_features.select(
    F.mean("current_week_points").alias("mean"),
    F.stddev("current_week_points").alias("std"),
    F.min("current_week_points").alias("min"),
    F.expr("percentile(current_week_points, 0.25)").alias("q25"),
    F.expr("percentile(current_week_points, 0.50)").alias("median"),
    F.expr("percentile(current_week_points, 0.75)").alias("q75"),
    F.max("current_week_points").alias("max")
).first()

print(f"  Mean: {target_stats['mean']:.2f}")
print(f"  Std Dev: {target_stats['std']:.2f}")
print(f"  Min: {target_stats['min']:.2f}")
print(f"  25th percentile: {target_stats['q25']:.2f}")
print(f"  Median: {target_stats['median']:.2f}")
print(f"  75th percentile: {target_stats['q75']:.2f}")
print(f"  Max: {target_stats['max']:.2f}")
print()

print("✓ Feature quality check complete")

In [0]:
print("Creating Train/Validation/Test Splits")
print("=" * 70)
print()

# Define split boundaries
# Training: 2024 Weeks 1-12
# Validation: 2024 Weeks 13-15
# Test: 2024 Weeks 16-18 + 2025 data

print("Split Strategy:")
print("  Training:   2024 Weeks 1-12")
print("  Validation: 2024 Weeks 13-15")
print("  Test:       2024 Weeks 16-18 + 2025")
print()

# Create splits
train_df = ml_features.filter(
    (F.col("season") == 2024) & (F.col("week") <= 12)
)

val_df = ml_features.filter(
    (F.col("season") == 2024) & (F.col("week").between(13, 15))
)

test_df = ml_features.filter(
    ((F.col("season") == 2024) & (F.col("week") >= 16)) |
    (F.col("season") == 2025)
)

print("Split Sizes:")
print(f"  Training:   {train_df.count():,} records")
print(f"  Validation: {val_df.count():,} records")
print(f"  Test:       {test_df.count():,} records")
print()

# Verify no overlap and temporal ordering
print("Split Verification:")
train_weeks = train_df.select("season", "week").distinct().orderBy("season", "week")
val_weeks = val_df.select("season", "week").distinct().orderBy("season", "week")
test_weeks = test_df.select("season", "week").distinct().orderBy("season", "week")

print(f"  Train weeks: {train_weeks.count()} unique season-week combinations")
print(f"  Val weeks: {val_weeks.count()} unique season-week combinations")
print(f"  Test weeks: {test_weeks.count()} unique season-week combinations")
print()

# Position distribution per split
print("Position Distribution per Split:")
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    pos_dist = split_df.groupBy("position").count().orderBy("position").collect()
    print(f"  {split_name}: " + ", ".join([f"{row['position']}={row['count']}" for row in pos_dist]))

print()
print("✓ Train/val/test splits created successfully")

In [0]:
print("Baseline Model 1: Season Average Prediction")
print("=" * 70)
print()
print("Predicting each player's points using their season average up to that week")
print()

# Calculate season average for each player up to each week (expanding window)
# This simulates what we'd know at prediction time

def calculate_expanding_avg(df, split_name):
    """
    Calculate expanding window average for each player.
    For week N, use average of weeks 1 to N-1 (exclude current week to avoid leakage).
    """
    window_spec = Window.partitionBy("master_player_id", "season").orderBy("week").rowsBetween(Window.unboundedPreceding, -1)
    
    df_with_pred = df.withColumn(
        "predicted_points_season_avg",
        F.avg("current_week_points").over(window_spec)
    )
    
    # Handle cases where there's no prior history (first game of season)
    # Use overall position average as fallback
    position_avg = df.groupBy("position").agg(
        F.mean("current_week_points").alias("position_avg")
    )
    
    df_with_pred = df_with_pred.join(position_avg, "position", "left")
    
    df_with_pred = df_with_pred.withColumn(
        "predicted_points_season_avg",
        F.coalesce(F.col("predicted_points_season_avg"), F.col("position_avg"))
    )
    
    return df_with_pred.drop("position_avg")

# Apply to each split
train_with_baseline1 = calculate_expanding_avg(train_df, "train")
val_with_baseline1 = calculate_expanding_avg(val_df, "val")
test_with_baseline1 = calculate_expanding_avg(test_df, "test")

print("✓ Season average predictions calculated for all splits")
print()

# Evaluate on validation set
print("Validation Set Evaluation (Baseline 1 - Season Average):")
print()

# Calculate metrics per position
for position in ["QB", "RB", "WR", "TE"]:
    pos_data = val_with_baseline1.filter(F.col("position") == position)
    
    metrics = pos_data.select(
        F.sqrt(F.mean(F.pow(F.col("current_week_points") - F.col("predicted_points_season_avg"), 2))).alias("rmse"),
        F.mean(F.abs(F.col("current_week_points") - F.col("predicted_points_season_avg"))).alias("mae"),
        F.count("*").alias("n")
    ).first()
    
    print(f"  {position}:")
    print(f"    RMSE: {metrics['rmse']:.2f} points")
    print(f"    MAE:  {metrics['mae']:.2f} points")
    print(f"    n:    {metrics['n']:,} predictions")
    print()

print("✓ Baseline 1 complete")

In [0]:
print("Baseline Model 2: Last 3 Games Average Prediction")
print("=" * 70)
print()
print("Predicting each player's points using their last 3 games average")
print()

def calculate_last_n_avg(df, n=3):
    """
    Calculate rolling N-game average for each player.
    For week N, use average of previous N weeks (exclude current week to avoid leakage).
    """
    window_spec = Window.partitionBy("master_player_id", "season").orderBy("week").rowsBetween(-n, -1)
    
    df_with_pred = df.withColumn(
        f"predicted_points_last{n}g_avg",
        F.avg("current_week_points").over(window_spec)
    )
    
    # Handle cases where there's insufficient history
    # Use season average as fallback
    season_window = Window.partitionBy("master_player_id", "season").orderBy("week").rowsBetween(Window.unboundedPreceding, -1)
    
    df_with_pred = df_with_pred.withColumn(
        "season_avg_fallback",
        F.avg("current_week_points").over(season_window)
    )
    
    # Use position average if no personal history
    position_avg = df.groupBy("position").agg(
        F.mean("current_week_points").alias("position_avg")
    )
    
    df_with_pred = df_with_pred.join(position_avg, "position", "left")
    
    df_with_pred = df_with_pred.withColumn(
        f"predicted_points_last{n}g_avg",
        F.coalesce(
            F.col(f"predicted_points_last{n}g_avg"),
            F.col("season_avg_fallback"),
            F.col("position_avg")
        )
    )
    
    return df_with_pred.drop("season_avg_fallback", "position_avg")

# Apply to each split
train_with_baseline2 = calculate_last_n_avg(train_df, n=3)
val_with_baseline2 = calculate_last_n_avg(val_df, n=3)
test_with_baseline2 = calculate_last_n_avg(test_df, n=3)

print("✓ Last 3 games average predictions calculated for all splits")
print()

# Evaluate on validation set
print("Validation Set Evaluation (Baseline 2 - Last 3 Games Average):")
print()

# Calculate metrics per position
for position in ["QB", "RB", "WR", "TE"]:
    pos_data = val_with_baseline2.filter(F.col("position") == position)
    
    metrics = pos_data.select(
        F.sqrt(F.mean(F.pow(F.col("current_week_points") - F.col("predicted_points_last3g_avg"), 2))).alias("rmse"),
        F.mean(F.abs(F.col("current_week_points") - F.col("predicted_points_last3g_avg"))).alias("mae"),
        F.count("*").alias("n")
    ).first()
    
    print(f"  {position}:")
    print(f"    RMSE: {metrics['rmse']:.2f} points")
    print(f"    MAE:  {metrics['mae']:.2f} points")
    print(f"    n:    {metrics['n']:,} predictions")
    print()

print("✓ Baseline 2 complete")

In [0]:
print("Baseline Models Comparison")
print("=" * 70)
print()
print("Comparing Season Average vs Last 3 Games Average on Validation Set")
print()

# Combine both baselines for comparison
comparison_results = []

for position in ["QB", "RB", "WR", "TE"]:
    # Season average
    pos_data1 = val_with_baseline1.filter(F.col("position") == position)
    metrics1 = pos_data1.select(
        F.sqrt(F.mean(F.pow(F.col("current_week_points") - F.col("predicted_points_season_avg"), 2))).alias("rmse")
    ).first()
    
    # Last 3 games
    pos_data2 = val_with_baseline2.filter(F.col("position") == position)
    metrics2 = pos_data2.select(
        F.sqrt(F.mean(F.pow(F.col("current_week_points") - F.col("predicted_points_last3g_avg"), 2))).alias("rmse")
    ).first()
    
    comparison_results.append({
        "position": position,
        "season_avg_rmse": metrics1["rmse"],
        "last3g_avg_rmse": metrics2["rmse"],
        "best_baseline": "Season Avg" if metrics1["rmse"] < metrics2["rmse"] else "Last 3G Avg",
        "best_rmse": min(metrics1["rmse"], metrics2["rmse"])
    })

from pyspark.sql import Row
comparison_df = spark.createDataFrame([Row(**r) for r in comparison_results])

print("Baseline RMSE Comparison:")
display(comparison_df)

print()
print("Summary:")
for result in comparison_results:
    print(f"  {result['position']}: Best baseline is {result['best_baseline']} with RMSE = {result['best_rmse']:.2f}")

print()
print("Target for ML Models:")
print("  Beat best baseline by at least 15% on RMSE")
for result in comparison_results:
    target_rmse = result['best_rmse'] * 0.85
    print(f"  {result['position']}: Target RMSE < {target_rmse:.2f} points")

print()
print("=" * 70)
print("✓ Phase 1 Complete: Data prepared and baselines established")
print("=" * 70)
print()
print("Next Steps:")
print("  - Phase 2: Train position-specific LightGBM models")
print("  - Phase 3: Hyperparameter tuning and model selection")
print("  - Phase 4: Model registry and deployment")